# Chatbot RAG — The Acts of the Apostles

## 1. Indexação dos embeddings com FAISS

Etapas:
1. Ler o `.txt` e separar o livro por capítulo (cada capítulo começa com `\fChap. N - Título`).
2. Quebrar cada capítulo em *chunks* menores.
3. Gerar os embeddings com `intfloat/multilingual-e5-small`, que é multilíngue: perguntas em português encontram trechos em inglês.
4. Indexar no FAISS e salvar o índice em disco.

In [46]:
import re
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

SOURCE_FILE = Path("rag-file/The-Acts-of-the-Apostles.txt")
INDEX_DIR = Path("faiss-index")
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"

### 1.1 Carregar o texto e separar por capítulo

No corpo do livro, cada capítulo começa com um *form feed* (`\f`) seguido de `Chap. N - Título`. O título pode ocupar duas linhas.
Tudo o que vem antes do capítulo 1 (licença, prefácio, sumário) fica de fora.
Cada quebra de página traz o número da página numa linha própria, seguido de `\f`. Esses números são removidos para não aparecerem no meio das frases.
O texto também tem quebras de linha no meio das frases, então os espaços em branco são normalizados.

In [47]:
text = SOURCE_FILE.read_text(encoding="utf-8")

# quebra da pagina
PAGE_BREAK_RE = re.compile(r"\n\s*\d{1,3}\s*(?:\f|$)")
# captura o cabecalho de cada capitulo e captura o numero e o tiulo
CHAPTER_RE = re.compile(r"\fChap\.\s*(\d+)\s*-\s*([^\n]+(?:\n[^\n]+)?)\n\n")
# lista dos 58 cabecalhos
matches = list(CHAPTER_RE.finditer(text))

chapters = []
for i, m in enumerate(matches):
    # index de onde o capitulo termina -> inicio do proximo cabecalho
    end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
    # recorte do final do cabecalho ate o inicio do outro
    body = PAGE_BREAK_RE.sub(" ", text[m.end():end])
    # limpa as quebras de pagina do body
    body = re.sub(r"\s+", " ", body).strip()
    chapters.append(Document(
        page_content=body,
        metadata={
            "chapter": int(m.group(1)), # pega o (\d+) do regex do chapter
            "title": re.sub(r"\s+", " ", m.group(2)).strip(), # pega o ([^\n]+(?:\n[^\n]+)?) do regear do chapter e limpa
            "source": SOURCE_FILE.name,
        },
    ))

print(f"{len(chapters)} capítulos")
chapters[0].metadata, chapters[0].page_content[:300]

58 capítulos


({'chapter': 1,
  'title': 'God’s Purpose for His Church',
  'source': 'The-Acts-of-the-Apostles.txt'},
 'The church is God’s appointed agency for the salvation of men. It was organized for service, and its mission is to carry the gospel to the world. From the beginning it has been God’s plan that through His church shall be reflected to the world His fullness and His sufficiency. The members of the chu')

### 1.2 Dividir em chunks

O `multilingual-e5-small` trunca a entrada em 512 *tokens*, ou seja, uns 2.000 caracteres em inglês.
Chunks de ~800 caracteres com 100 de sobreposição ficam bem abaixo desse limite.
Cada chunk herda o capítulo e o título como metadados.

In [48]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=[". ", "? ", "! ", "; ", ", ", " ", ""],
    keep_separator="end",
)
chunks = splitter.split_documents(chapters)

print(f"{len(chunks)} chunks")
chunks[0]

1317 chunks


Document(metadata={'chapter': 1, 'title': 'God’s Purpose for His Church', 'source': 'The-Acts-of-the-Apostles.txt'}, page_content='The church is God’s appointed agency for the salvation of men. It was organized for service, and its mission is to carry the gospel to the world. From the beginning it has been God’s plan that through His church shall be reflected to the world His fullness and His sufficiency. The members of the church, those whom He has called out of darkness into His marvelous light, are to show forth His glory. The church is the repository of the riches of the grace of Christ; and through the church will eventually be made manifest, even to “the principalities and powers in heavenly places,” the final and full display of the love of God. Ephesians 3:10. Many and wonderful are the promises recorded in the Scriptures regarding the church.')

### 1.3 Gerar os embeddings e indexar no FAISS

Os modelos E5 foram treinados com prefixos: `"passage: "` nos trechos indexados e `"query: "` nas perguntas. Sem eles, a qualidade da busca cai.
O `prompt` do sentence-transformers acrescenta o prefixo automaticamente: `encode_kwargs` vale para os documentos e `query_encode_kwargs` para as perguntas.

Com `normalize_embeddings=True`, ordenar pela distância L2 dá o mesmo resultado que ordenar pela similaridade de cosseno.

In [49]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True, "prompt": "passage: "},
    query_encode_kwargs={"normalize_embeddings": True, "prompt": "query: "},
)

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(str(INDEX_DIR))

print(f"{vectorstore.index.ntotal} vetores de dimensão {vectorstore.index.d} salvos em {INDEX_DIR}/")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19846.08it/s]


1317 vetores de dimensão 384 salvos em faiss-index/


### 1.4 Testar o retrieval

Depois de salvo, o índice pode ser recarregado sem gerar os embeddings de novo.
`allow_dangerous_deserialization` é necessário porque o docstore é salvo em pickle; só use com índices gerados por você.

In [50]:
vectorstore = FAISS.load_local(
    str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True
)

for query in ["What happened with Stephen?", "O que aconteceu com Estêvão?"]:
    print(f"=== {query}")
    for doc, score in vectorstore.similarity_search_with_score(query, k=4):
        print(f"[{score:.3f}] Cap. {doc.metadata['chapter']} - {doc.metadata['title']}")
        print(doc.page_content[:250], "...\n")

=== What happened with Stephen?
[0.299] Cap. 18 - Preaching Among the Heathen
Soon the murderous designs of the enemies of the gospel were carried out. Yielding to the influence of evil, the Lystrians became possessed with a satanic fury and, seizing Paul, mercilessly stoned him. The apostle thought that his end had come. The  ...

[0.311] Cap. 10 - The First Christian Martyr
Stephen, the foremost of the seven deacons, was a man of deep piety and broad faith. Though a Jew by birth, he spoke the Greek language and was familiar with the customs and manners of the Greeks. He therefore found opportunity to preach the gospel i ...

[0.311] Cap. 10 - The First Christian Martyr
Stopping their ears, that they might not hear his words, and uttering loud cries, they ran furiously upon him with one accord “and cast him out of the city.” “And they stoned Stephen, calling upon God, and saying, Lord Jesus, receive my spirit. And h ...

[0.324] Cap. 10 - The First Christian Martyr
He connected Jesus 

## 2. Gerar a resposta com o `flan-t5-base`

O `google/flan-t5-base` é um modelo *seq2seq* (encoder-decoder) do Google, com ~250 milhões de parâmetros, treinado para seguir instruções curtas.
Ele roda localmente, sem API, mas tem limites:
- entende bem só **inglês**;
- foi treinado com entradas de até **512 tokens**. Os 4 chunks somam ~750 tokens; o T5 aceita entradas maiores porque usa posições relativas, mas a qualidade pode cair;
- dá respostas **curtas e extrativas**, geralmente copiando um pedaço do contexto.

### 2.1 Carregar o modelo

In [79]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GENERATOR_MODEL = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
generator = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_MODEL)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 14687.62it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


### 2.2 Recuperar os 4 trechos mais parecidos e responder

`similarity_search_with_score(question, k=4)` devolve os 4 chunks de **menor score**. Os textos são juntados num único contexto, e o modelo é instruído a responder em duas ou três frases.

Dois detalhes do prompt:
- **A pergunta vem antes do contexto.** Se a entrada passar do `max_length`, o tokenizer corta o final; assim se perde o fim do último trecho, e não a pergunta.
- **`max_length=1024`** para caber os 4 trechos (~750 tokens) sem truncar.

Na geração, `num_beams=4` explora 4 sequências candidatas em paralelo e fica com a mais provável, e `no_repeat_ngram_size=3` impede que o modelo repita a mesma sequência de 3 tokens.

In [82]:
def answer(question):
    # k=4: os 4 trechos de menor score, ou seja, os mais parecidos com a pergunta
    context = vectorstore.similarity_search_with_score(question, k=1)

    # a pergunta vem primeiro: se o prompt for truncado, perde-se o fim do contexto, não a pergunta
    prompt = (
        f"Question: {question}\n\n"
        "Answer the question in two or three sentences, using only the context below.\n\n"
        f"Context: {context}"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    output = generator.generate(
        **inputs, max_new_tokens=120, num_beams=4, no_repeat_ngram_size=3
    )

    return tokenizer.decode(output[0], skip_special_tokens=True), context

### 2.3 Testar

In [83]:
for question in [
    "What happened with Stephen?",
    "How many souls were added to the church at Pentecost?",
    "Who freed Peter from prison?",
    "What did Paul do in Athens?",
    "O que aconteceu com Estêvão?",
]:
    response, results = answer(question)
    print(f"Pergunta: {question}")
    print(f"Resposta: {response}")
    print("Fontes:")
    for doc, score in results:
        print(f"  [{score:.3f}] Cap. {doc.metadata['chapter']} - {doc.metadata['title']}")
    print()

Pergunta: What happened with Stephen?
Resposta: he fell to the ground, and the infuriated mob “drew him out of the city, supposing he had been dead
Fontes:
  [0.299] Cap. 18 - Preaching Among the Heathen

Pergunta: How many souls were added to the church at Pentecost?
Resposta: about three thousand souls
Fontes:
  [0.275] Cap. 4 - Pentecost

Pergunta: Who freed Peter from prison?
Resposta: God
Fontes:
  [0.300] Cap. 15 - Delivered From Prison

Pergunta: What did Paul do in Athens?
Resposta: looked upon the beauty and grandeur surrounding him, and saw the city wholly given to idolatry, his spirit was stirred with jealousy for God, whom he saw dishonored on every side, and his heart was drawn out in pity for the people of Athens, who, notwithstanding their intellectual culture, were ignorant of the true God
Fontes:
  [0.276] Cap. 23 - Berea and Athens

Pergunta: O que aconteceu com Estêvão?
Resposta: Their resurrection and ascension, and His work in heaven as the Mediator for fallen man,

## 3. Lang Chain

## 3. RAG com o Claude (Anthropic) via LangChain

O `flan-t5-base` só funciona bem em inglês e copia trechos em vez de sintetizar. Aqui o gerador passa a ser o `claude-opus-5-5`, chamado pelo `ChatAnthropic` do `langchain-anthropic`.
Ele junta os 4 trechos recuperados numa resposta, responde na língua da pergunta e cita os capítulos.

### 3.1 Configurar o modelo

- A chave `ANTHROPIC_API_KEY` vem do `.env`, carregado pelo `load_dotenv()`.
- `effort` controla quanto o modelo "pensa" antes de responder. Para perguntas sobre um contexto curto, `low` basta e é mais barato e rápido.
- `fallbacks: "default"`: se um classificador de segurança recusar a pergunta, a própria API tenta de novo num modelo alternativo, sem código extra.

In [84]:
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()  # lê ANTHROPIC_API_KEY do .env

llm = ChatAnthropic(
    model="claude-opus-5-5",
    max_tokens=4000,
    # perguntas e respostas sobre um contexto curto: esforço baixo basta e sai mais barato
    output_config={"effort": "low"},
    # se um classificador de segurança recusar a pergunta, a API tenta de novo em outro modelo
    betas=["server-side-fallback-2026-07-01"],
    model_kwargs={"fallbacks": "default"},
)

### 3.2 Montar a chain

A chain é escrita em LCEL (o operador `|` do LangChain encadeia as etapas):

1. `retriever` busca os 4 chunks mais parecidos com a pergunta no FAISS;
2. `format_docs` junta os chunks num texto só, cada um precedido pelo capítulo de onde veio;
3. `prompt` coloca esse texto no *system prompt*, dentro de `<context>`, e a pergunta na mensagem do usuário;
4. `llm` chama o Claude;
5. `StrOutputParser` extrai só o texto da resposta.

`RunnablePassthrough()` repassa a pergunta original sem mudanças para o campo `question` do prompt.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


def format_docs(docs):
    # cada trecho vai com o capítulo de onde veio, para o Claude poder citar a fonte
    return "\n\n".join(
        f"[Cap. {doc.metadata['chapter']} - {doc.metadata['title']}]\n{doc.page_content}"
        for doc in docs
    )


prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about the book \"The Acts of the Apostles\" by Ellen G. White.\n"
     "Use only the passages in <context>. If they don't contain the answer, say that you "
     "couldn't find it in the book; don't answer from general knowledge.\n" "Answer in the same language as the question, in a short paragraph, and cite the chapters "
     "you used, e.g. (Cap. 10).\n\n"
     "<context>\n{context}\n</context>"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### 3.3 Testar

In [86]:
for question in [
    "What happened with Stephen?",
    "Quem libertou Pedro da prisão?",
    "O que aconteceu com Estêvão?",
]:
    print(f"Pergunta: {question}")
    print(f"Resposta: {rag_chain.invoke(question)}\n")

Pergunta: What happened with Stephen?
Resposta: Stephen was the foremost of the seven deacons, a man of deep piety and faith who spoke Greek and preached the gospel in the synagogues of the Greek Jews. Learned rabbis and doctors of the law debated him publicly, but they could not resist the wisdom and spirit with which he spoke. In his final testimony he connected Jesus Christ with all of Jewish history and with the prophecies, and he spoke of the temple, saying that "the Most High dwelleth not in temples made with hands." At this the people grew agitated and the priest tore his robe, and Stephen knew he was giving his last testimony. His hearers stopped their ears, rushed at him, cast him out of the city, and stoned him. As he died he prayed, "Lord Jesus, receive my spirit," and "Lord, lay not this sin to their charge." No legal sentence had been passed on him, and the Roman authorities were bribed not to investigate. His martyrdom deeply impressed everyone who saw it (Cap. 10). Paul,

In [87]:
question="Who was Gammaliel?"

print(f"Pergunta: {question}")
print(f"Resposta: {rag_chain.invoke(question)}\n")

Pergunta: Who was Gammaliel?
Resposta: Gamaliel was a Pharisee of good reputation, a learned man of high position who sat on the Jewish council (the Sanhedrin). When the priests wanted to put the apostles to death, he recognized "the voice of God" in the disciples' words. He saw that such a violent step would lead to terrible consequences. He had the prisoners removed and then calmly advised the council to leave them alone. He pointed to failed movements like those of Theudas and Judas of Galilee: if this work was of men, it would come to nothing, but if it was of God, they could not overthrow it and might even be found fighting against God. The priests were obliged to agree with him (Cap. 8). He was also Paul's teacher. In his defense in Jerusalem, Paul said he was "brought up in this city at the feet of Gamaliel" and taught according to the law of the fathers (Cap. 38).



In [88]:
question="What are the cities mentioned in the book?"

print(f"Pergunta: {question}")
print(f"Resposta: {rag_chain.invoke(question)}\n")

Pergunta: What are the cities mentioned in the book?
Resposta: I can only see a few excerpts, not the whole book, so this is a partial list. The cities named in these passages are **Ephesus**, the capital of the Roman province of Asia and a great commercial center; **Corinth**, where Apollos was preaching; **Jerusalem**; and **Antioch**, the scene of Paul's early work (Cap. 27). Antioch is also described as the metropolis of Syria, where the gospel was gladly received (Cap. 16). **Damascus** is where Saul offered to carry out the persecution of believers (Cap. 12), and **Tarsus**, "a city in Cilicia," is Paul's birthplace (Cap. 38). Other places mentioned, such as Phenice, Cyprus, Galatia, Phrygia and Asia Minor, are regions or islands rather than cities (Cap. 16, Cap. 27).



In [89]:
question="Return a table with the name of the cities mentioned in the book and the chapter where they are mentioned for the first time."

print(f"Pergunta: {question}")
print(f"Resposta: {rag_chain.invoke(question)}\n")

Pergunta: Return a table with the name of the cities mentioned in the book and the chapter where they are mentioned for the first time.
Resposta: I can't give the true first mention of each city in the whole book, because I only have four short passages (Cap. 21, 24, 27 and 57). Some cities, like Jerusalem and Antioch, almost certainly appear in earlier chapters I can't see. The table below shows only the **earliest chapter, among these passages**, where each city appears.

| City | First chapter in these passages |
|---|---|
| Troas | Cap. 21 |
| Neapolis | Cap. 21 |
| Philippi | Cap. 21 |
| Thyatira | Cap. 21 |
| Corinth | Cap. 24 |
| Ephesus | Cap. 27 |
| Jerusalem | Cap. 27 |
| Antioch | Cap. 27 |
| Smyrna | Cap. 57 |
| Pergamos | Cap. 57 |
| Sardis | Cap. 57 |
| Philadelphia | Cap. 57 |
| Laodicea | Cap. 57 |

Cap. 21 also mentions Samothracia, but that is an island, not a city, so I left it out of the table.

